# FINAL — Generate Dataset Gabungan, Grid Search, dan Test Per Skenario CNN / CNN-LSTM

Notebook ini disusun agar dapat dijalankan **sekali dari atas ke bawah** di Google Colab.

Tahap yang dijalankan:

1. Generate seluruh dataset untuk:
   - Noise Varying
   - Fading Varying
   - Noise and Fading Varying
2. Generate dataset untuk tiga band deteksi:
   - PU 1 / `pu1_low`
   - PU 2 / `pu2_mid`
   - PU 3 / `pu3_high`
3. Simpan dataset CNN dan CNN-LSTM secara terpisah per skenario.
4. Gabungkan seluruh dataset besar untuk grid search.
5. Split dataset gabungan menjadi **70% train, 10% validation, dan 20% test**.
6. Simpan `idx_train`, `idx_val`, dan `idx_test` agar test set tidak perlu direkonstruksi.
7. Jalankan grid search CNN dan CNN-LSTM.
8. Simpan model terbaik, hasil grid search, evaluasi test global, dan metrik test per skenario.

Konfigurasi default menghasilkan:

```text
18 skenario kondisi × 3 PU = 54 skenario dataset
```

Setiap skenario menghasilkan:

```text
CNN      = 2000 data = 1000 PU + noise + 1000 noise only
CNN-LSTM = 2000 data = 1000 PU + noise + 1000 noise only
```

Total representasi data tersimpan:

```text
CNN      = 108.000 data = 54.000 PU + noise + 54.000 noise only
CNN-LSTM = 108.000 data = 54.000 PU + noise + 54.000 noise only
Total    = 216.000 representasi data
```

Metrik per skenario dihitung menggunakan **test set yang sama persis** dengan test set hasil split saat training, bukan split rekonstruksi dari notebook terpisah.


## 1. Import dan Random Seed


In [ ]:
import os
import json
import math
import random
import gc
from pathlib import Path
from itertools import product
from sklearn.metrics import roc_curve, auc

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.linalg import convolution_matrix

from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as err:
    print("Determinism tidak dapat diaktifkan penuh:", err)


## 2. Konfigurasi Umum

Ubah `BASE_DIR` sesuai lokasi penyimpanan notebook dan file `fading-10.csv` sampai `fading-15.csv`.

Contoh jika memakai Google Drive di Colab:

```python
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/nama_folder_dataset')
```

Jika file CSV fading diunggah langsung ke Colab, `BASE_DIR = Path('.')` sudah cukup.


In [ ]:
BASE_DIR = Path("dataset")
OUTPUT_DIR = BASE_DIR / "generated_datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Proteksi file output.
# False: jika file .npz sudah ada, skenario tersebut akan dilewati agar tidak tertimpa.
# True : file lama akan ditimpa.
OVERWRITE_DATASETS = False

COMMON_CONFIG = {
    "num_samples": 1000,
    "N1": 1000,
    "N2": 1000,
    "unfiltered_signal_length": 10000,
    "mean": 0,
    "unfiltered_signal_var": 80,
    "total_antennas": 6,
    "window_length": 1000,
    "hop": 1000,
    "divide_matrix": True,
    "corrcoef": False,
    "dtype": "float32",
}

# Tiga PU yang dibangkitkan pada sinyal sumber.
theta_ranges_1 = [
    (np.pi * 0.1, np.pi * 0.25),  # PU 1
    (np.pi * 0.4, np.pi * 0.55),  # PU 2
    (np.pi * 0.7, np.pi * 0.85),  # PU 3
]

# Tiga band penerima/detektor. Dataset akan dibuat untuk semuanya.
RECEIVER_BANDS = {
    "pu1_low":  [(np.pi * 0.08,  np.pi * 0.252)],
    "pu2_mid":  [(np.pi * 0.398, np.pi * 0.552)],
    "pu3_high": [(np.pi * 0.698, np.pi * 0.852)],
}

# Jika hanya ingin membuat sebagian PU, isi daftar ini, misalnya ["pu3_high"].
# Default None berarti seluruh PU 1, PU 2, dan PU 3 diproses.
ACTIVE_RECEIVER_BANDS = None


## 3. Daftar Skenario Dataset

Bagian ini mendefinisikan seluruh skenario yang akan dibuat.

- `noise_varying`: seluruh list `noise_var` dari notebook noise varying.
- `fading_varying`: seluruh file fading `fading-10.csv` sampai `fading-15.csv`, dengan noise tetap.
- `noise_and_fading_varying`: seluruh file fading `fading-10.csv` sampai `fading-15.csv`, dengan noise yang juga berubah antar-window.

Jika daftar skenario lama Anda berbeda, cukup ubah list di cell ini.


In [ ]:
NOISE_VARYING_SCENARIOS = {
    "noise_100_190": [100, 110, 120, 130, 140, 150, 160, 170, 180, 190],
    "noise_200_290": [200, 210, 220, 230, 240, 250, 260, 270, 280, 290],
    "noise_300_390": [300, 310, 320, 330, 340, 350, 360, 370, 380, 390],
    "noise_400_490": [400, 410, 420, 430, 440, 450, 460, 470, 480, 490],
    "noise_500_590": [500, 510, 520, 530, 540, 550, 560, 570, 580, 590],
    "noise_600_690": [600, 610, 620, 630, 640, 650, 660, 670, 680, 690],
}

# Alternatif skenario halus dari komentar notebook lama.
FINE_NOISE_VARYING_SCENARIOS = {
    "noise_280_297_1": [280, 281.9, 283.8, 285.7, 287.6, 289.5, 291.4, 293.3, 295.2, 297.1],
    "noise_300_317_1": [300, 301.9, 303.8, 305.7, 307.6, 309.5, 311.4, 313.3, 315.2, 317.1],
    "noise_320_337_1": [320, 321.9, 323.8, 325.7, 327.6, 329.5, 331.4, 333.3, 335.2, 337.1],
    "noise_360_377_1": [360, 361.9, 363.8, 365.7, 367.6, 369.5, 371.4, 373.3, 375.2, 377.1],
    "noise_380_397_1": [380, 381.9, 383.8, 385.7, 387.6, 389.5, 391.4, 393.3, 395.2, 397.1],
    "noise_400_417_1": [400, 401.9, 403.8, 405.7, 407.6, 409.5, 411.4, 413.3, 415.2, 417.1],
}

# Gunakan ini jika skenario halus juga ingin dimasukkan.
USE_FINE_NOISE_SCENARIOS = False
if USE_FINE_NOISE_SCENARIOS:
    NOISE_VARYING_SCENARIOS = {**NOISE_VARYING_SCENARIOS, **FINE_NOISE_VARYING_SCENARIOS}

FADING_FILES = [f"fading-{i}.csv" for i in range(10, 16)]

# Nilai noise tetap untuk FTV mengikuti notebook fading varying.
FIXED_NOISE_VAR_FOR_FTV = 80

# Nilai noise varying untuk NTV + FTV mengikuti notebook noise and fading varying.
NOISE_VAR_FOR_NTV_FTV = [90, 96, 102, 108, 114, 120, 126, 132, 138, 144]

# Fading tetap untuk NTV mengikuti notebook noise varying.
FIXED_FADING_FOR_NTV = np.array([
    0.42022108,
    -0.26307584,
    -0.44580676,
    -0.11005037,
    0.20913527,
    -0.27809250,
], dtype=np.float64)


## 4. Fungsi Sinyal, Noise, Fading, dan Matriks Kovarian


In [ ]:
def create_bpf(theta_1, theta_2, N):
    bpf = np.zeros(N)
    for i in range(N):
        bpf[i] = (
            (theta_2 / np.pi) * np.sinc(theta_2 * (i - 0.5 * N) / np.pi)
            - (theta_1 / np.pi) * np.sinc(theta_1 * (i - 0.5 * N) / np.pi)
        )
    return bpf


def generate_pu_signal(theta_ranges, mean, unfiltered_signal_var, unfiltered_signal_length, N, total_antennas):
    rand_sig = np.random.normal(mean, math.sqrt(unfiltered_signal_var), unfiltered_signal_length)
    pu_total = np.zeros(unfiltered_signal_length)

    for theta1, theta2 in theta_ranges:
        bpf = create_bpf(theta1, theta2, N)
        filtered = np.convolve(rand_sig, bpf, mode="same")
        pu_total += filtered

    return np.tile(pu_total, (total_antennas, 1))


def matrix_divider(signal_matrix):
    signal_odd = signal_matrix[:, ::2]
    signal_even = signal_matrix[:, 1::2]
    return np.vstack((signal_even, signal_odd))


def generate_gaussian_noise(n, mean, noise_var, length):
    return np.random.normal(mean, math.sqrt(float(noise_var)), size=(n, length))


def load_fading_csv(csv_path):
    df = pd.read_csv(csv_path)
    fading = np.array([df[col].to_numpy() for col in df.columns], dtype=np.float64)

    # Format yang umum dari notebook lama adalah (jumlah_window, total_antennas).
    # Jika CSV terbaca sebagai (total_antennas, jumlah_window), transpose otomatis.
    if fading.shape[0] == COMMON_CONFIG["total_antennas"] and fading.shape[1] != COMMON_CONFIG["total_antennas"]:
        fading = fading.T

    return fading


def get_noise_for_window(noise_var, window_index):
    if np.isscalar(noise_var):
        return float(noise_var)
    return float(noise_var[window_index])


def get_fading_for_window(fading, window_index, total_antennas):
    fading = np.asarray(fading)

    if fading.ndim == 1:
        if fading.shape[0] != total_antennas:
            raise ValueError(f"Fading tetap harus berukuran ({total_antennas},), tetapi diperoleh {fading.shape}")
        return fading

    if fading.ndim == 2:
        if fading.shape[0] <= window_index:
            raise ValueError(f"Jumlah baris fading ({fading.shape[0]}) lebih kecil dari jumlah window yang dibutuhkan.")
        if fading.shape[1] != total_antennas:
            raise ValueError(f"Fading varying harus berukuran (jumlah_window, {total_antennas}), tetapi diperoleh {fading.shape}")
        return fading[window_index]

    raise ValueError(f"Format fading tidak dikenali: {fading.shape}")


def apply_fading(segment, fading_vector):
    # Pada notebook lama fading digunakan melalui np.convolve.
    # Jika nilai fading adalah skalar per antena, operasi tersebut ekuivalen dengan perkalian amplitudo.
    return segment * fading_vector[:, np.newaxis]


def build_receiver_filter(theta_ranges_2, N2, window_length):
    bpf_total = np.zeros(N2)
    for theta_1, theta_2 in theta_ranges_2:
        bpf_total += create_bpf(theta_1, theta_2, N2)
    return convolution_matrix(bpf_total, window_length, mode="same")


def covariance_matrix(signal_matrix, divide_matrix=True, corrcoef=False):
    if divide_matrix:
        signal_matrix = matrix_divider(signal_matrix)

    if corrcoef:
        return np.corrcoef(signal_matrix)

    centered = signal_matrix - signal_matrix.mean(axis=1, keepdims=True)
    return (centered @ centered.T) / centered.shape[1]


## 5. Fungsi Pembentukan Dataset CNN dan CNN-LSTM


In [ ]:
def generate_windowed_data(signal, window_length, hop, mean, noise_var, fading):
    A, T = signal.shape
    starts = list(range(0, T - window_length + 1, hop))

    if not np.isscalar(noise_var) and len(noise_var) != len(starts):
        raise ValueError(f"noise_var harus skalar atau berisi {len(starts)} nilai, tetapi diperoleh {len(noise_var)} nilai.")

    windows_pu = []
    windows_noise = []

    for window_index, start in enumerate(starts):
        pu_segment = signal[:, start:start + window_length]

        current_noise_var = get_noise_for_window(noise_var, window_index)
        noise_full = generate_gaussian_noise(A, mean, current_noise_var, T)
        noise_segment = noise_full[:, start:start + window_length]

        fading_vector = get_fading_for_window(fading, window_index, A)
        faded_segment = apply_fading(pu_segment, fading_vector)

        windows_pu.append(faded_segment + noise_segment)
        windows_noise.append(noise_segment)

    return windows_pu, windows_noise


def filter_windows(windows, H):
    W = len(windows)
    A = windows[0].shape[0]

    columns = []
    for w in range(W):
        for a in range(A):
            columns.append(windows[w][a])

    X = np.column_stack(columns)
    Y = H @ X

    windows_filtered = []
    idx = 0
    for w in range(W):
        seg = []
        for a in range(A):
            seg.append(Y[:, idx])
            idx += 1
        windows_filtered.append(np.array(seg))

    return windows_filtered


def make_cnn_matrix(windows, H, divide_matrix=True, corrcoef=False):
    windows_filtered = filter_windows(windows, H)
    signal_filtered = np.hstack(windows_filtered)
    return covariance_matrix(signal_filtered, divide_matrix=divide_matrix, corrcoef=corrcoef)


def make_cnn_lstm_sequence(windows, H, divide_matrix=True, corrcoef=False):
    windows_filtered = filter_windows(windows, H)
    corr_seq = []
    for seg in windows_filtered:
        corr_seq.append(covariance_matrix(seg, divide_matrix=divide_matrix, corrcoef=corrcoef))
    return np.array(corr_seq)


def create_dataset_pair(
    num_samples,
    N1,
    N2,
    unfiltered_signal_length,
    mean,
    unfiltered_signal_var,
    noise_var,
    total_antennas,
    theta_ranges_1,
    theta_ranges_2,
    window_length,
    hop,
    fading,
    divide_matrix=True,
    corrcoef=False,
    dtype="float32",
):
    H = build_receiver_filter(theta_ranges_2, N2, window_length)

    X_cnn, y_cnn = [], []
    X_cnn_lstm, y_cnn_lstm = [], []

    for sample_index in range(num_samples):
        if (sample_index + 1) % 100 == 0:
            print(f"  sample {sample_index + 1}/{num_samples}")

        PU_Signal = generate_pu_signal(
            theta_ranges_1,
            mean,
            unfiltered_signal_var,
            unfiltered_signal_length,
            N1,
            total_antennas,
        )

        shared_windows_pu, shared_windows_noise = generate_windowed_data(
            PU_Signal,
            window_length,
            hop,
            mean,
            noise_var,
            fading,
        )

        corr_pu_cnn = make_cnn_matrix(shared_windows_pu, H, divide_matrix=divide_matrix, corrcoef=corrcoef)
        corr_noise_cnn = make_cnn_matrix(shared_windows_noise, H, divide_matrix=divide_matrix, corrcoef=corrcoef)

        corr_pu_cnn_lstm = make_cnn_lstm_sequence(shared_windows_pu, H, divide_matrix=divide_matrix, corrcoef=corrcoef)
        corr_noise_cnn_lstm = make_cnn_lstm_sequence(shared_windows_noise, H, divide_matrix=divide_matrix, corrcoef=corrcoef)

        X_cnn.extend([corr_pu_cnn, corr_noise_cnn])
        y_cnn.extend([1, 0])

        X_cnn_lstm.extend([corr_pu_cnn_lstm, corr_noise_cnn_lstm])
        y_cnn_lstm.extend([1, 0])

    X_cnn = np.asarray(X_cnn, dtype=dtype)[..., np.newaxis]
    y_cnn = np.asarray(y_cnn, dtype=np.int64)

    X_cnn_lstm = np.asarray(X_cnn_lstm, dtype=dtype)[..., np.newaxis]
    y_cnn_lstm = np.asarray(y_cnn_lstm, dtype=np.int64)

    return {
        "cnn": {"X": X_cnn, "y": y_cnn},
        "cnn_lstm": {"X": X_cnn_lstm, "y": y_cnn_lstm},
    }


## 6. Membuat Seluruh Dataset dan Menyimpannya ke `.npz`

Cell ini membuat dataset per-skenario dan menyimpannya terpisah. Pola ini lebih aman dibanding menggabungkan semua skenario langsung ke RAM, karena ukuran dataset CNN-LSTM dapat besar.


In [ ]:
def dataset_output_paths(metadata, output_dir=OUTPUT_DIR):
    scenario_name = metadata["scenario_name"]
    scenario_type = metadata["scenario_type"]
    receiver_band = metadata["receiver_band"]

    cnn_path = output_dir / f"{receiver_band}__{scenario_type}__{scenario_name}__cnn.npz"
    cnn_lstm_path = output_dir / f"{receiver_band}__{scenario_type}__{scenario_name}__cnn_lstm.npz"
    return cnn_path, cnn_lstm_path


def save_dataset_npz(dataset, metadata, output_dir=OUTPUT_DIR, overwrite=OVERWRITE_DATASETS):
    cnn_path, cnn_lstm_path = dataset_output_paths(metadata, output_dir=output_dir)

    if not overwrite and cnn_path.exists() and cnn_lstm_path.exists():
        print("  SKIP SAVE: file sudah ada dan OVERWRITE_DATASETS=False")
        print("  CNN     :", cnn_path)
        print("  CNN-LSTM:", cnn_lstm_path)
        return cnn_path, cnn_lstm_path, True

    np.savez_compressed(
        cnn_path,
        X=dataset["cnn"]["X"],
        y=dataset["cnn"]["y"],
        metadata=json.dumps(metadata),
    )

    np.savez_compressed(
        cnn_lstm_path,
        X=dataset["cnn_lstm"]["X"],
        y=dataset["cnn_lstm"]["y"],
        metadata=json.dumps(metadata),
    )

    return cnn_path, cnn_lstm_path, False


def make_metadata(scenario_type, scenario_name, noise_var, fading_source, receiver_band, theta_ranges_2):
    return {
        "scenario_type": scenario_type,
        "scenario_name": scenario_name,
        "noise_var": noise_var if np.isscalar(noise_var) else list(map(float, noise_var)),
        "fading_source": fading_source,
        "receiver_band": receiver_band,
        "theta_ranges_1": [(float(a), float(b)) for a, b in theta_ranges_1],
        "theta_ranges_2": [(float(a), float(b)) for a, b in theta_ranges_2],
        **{k: (str(v) if k == "dtype" else v) for k, v in COMMON_CONFIG.items()},
    }


def generate_one_scenario(
    scenario_type,
    scenario_name,
    noise_var,
    fading,
    fading_source,
    receiver_band,
    theta_ranges_2,
):
    metadata = make_metadata(
        scenario_type=scenario_type,
        scenario_name=scenario_name,
        noise_var=noise_var,
        fading_source=fading_source,
        receiver_band=receiver_band,
        theta_ranges_2=theta_ranges_2,
    )
    cnn_path, cnn_lstm_path = dataset_output_paths(metadata)

    if not OVERWRITE_DATASETS and cnn_path.exists() and cnn_lstm_path.exists():
        print(f"\n=== SKIP {receiver_band} / {scenario_type} / {scenario_name}: file sudah ada ===")
        return {
            "receiver_band": receiver_band,
            "scenario_type": scenario_type,
            "scenario_name": scenario_name,
            "cnn_path": str(cnn_path),
            "cnn_lstm_path": str(cnn_lstm_path),
            "metadata": metadata,
            "status": "skipped_existing",
        }

    print(f"\n=== Membuat {receiver_band} / {scenario_type} / {scenario_name} ===")
    dataset = create_dataset_pair(
        num_samples=COMMON_CONFIG["num_samples"],
        N1=COMMON_CONFIG["N1"],
        N2=COMMON_CONFIG["N2"],
        unfiltered_signal_length=COMMON_CONFIG["unfiltered_signal_length"],
        mean=COMMON_CONFIG["mean"],
        unfiltered_signal_var=COMMON_CONFIG["unfiltered_signal_var"],
        noise_var=noise_var,
        total_antennas=COMMON_CONFIG["total_antennas"],
        theta_ranges_1=theta_ranges_1,
        theta_ranges_2=theta_ranges_2,
        window_length=COMMON_CONFIG["window_length"],
        hop=COMMON_CONFIG["hop"],
        fading=fading,
        divide_matrix=COMMON_CONFIG["divide_matrix"],
        corrcoef=COMMON_CONFIG["corrcoef"],
        dtype=COMMON_CONFIG["dtype"],
    )

    cnn_path, cnn_lstm_path, skipped = save_dataset_npz(dataset, metadata)
    print("  CNN     :", cnn_path)
    print("  CNN-LSTM:", cnn_lstm_path)
    print("  shape CNN     :", dataset["cnn"]["X"].shape, dataset["cnn"]["y"].shape)
    print("  shape CNN-LSTM:", dataset["cnn_lstm"]["X"].shape, dataset["cnn_lstm"]["y"].shape)

    return {
        "receiver_band": receiver_band,
        "scenario_type": scenario_type,
        "scenario_name": scenario_name,
        "cnn_path": str(cnn_path),
        "cnn_lstm_path": str(cnn_lstm_path),
        "metadata": metadata,
        "status": "generated" if not skipped else "skipped_existing",
    }


def selected_receiver_bands():
    if ACTIVE_RECEIVER_BANDS is None:
        return RECEIVER_BANDS

    missing = [band for band in ACTIVE_RECEIVER_BANDS if band not in RECEIVER_BANDS]
    if missing:
        raise ValueError(f"Receiver band tidak ditemukan: {missing}")

    return {band: RECEIVER_BANDS[band] for band in ACTIVE_RECEIVER_BANDS}


def expected_dataset_count():
    receiver_count = len(selected_receiver_bands())
    condition_count = len(NOISE_VARYING_SCENARIOS) + len(FADING_FILES) + len(FADING_FILES)
    scenario_count = receiver_count * condition_count
    samples_per_model = scenario_count * COMMON_CONFIG["num_samples"] * 2
    return {
        "receiver_count": receiver_count,
        "condition_count_per_receiver": condition_count,
        "scenario_count": scenario_count,
        "cnn_samples": samples_per_model,
        "cnn_lstm_samples": samples_per_model,
        "total_saved_representations": samples_per_model * 2,
        "npz_files": scenario_count * 2,
    }


def print_dataset_plan(minutes_per_scenario=2.5):
    info = expected_dataset_count()
    total_minutes = info["scenario_count"] * minutes_per_scenario
    print("Rencana dataset:")
    print(f"  Receiver band              : {info['receiver_count']}")
    print(f"  Skenario kondisi per PU     : {info['condition_count_per_receiver']}")
    print(f"  Total skenario dataset      : {info['scenario_count']}")
    print(f"  Data CNN                    : {info['cnn_samples']:,}".replace(',', '.'))
    print(f"  Data CNN-LSTM               : {info['cnn_lstm_samples']:,}".replace(',', '.'))
    print(f"  Total representasi tersimpan: {info['total_saved_representations']:,}".replace(',', '.'))
    print(f"  Total file .npz             : {info['npz_files']}")
    print(f"  Estimasi waktu generate     : {total_minutes:.1f} menit / {total_minutes/60:.2f} jam")
    return info


def generate_all_datasets():
    manifest = []
    receiver_band_items = selected_receiver_bands().items()

    for receiver_band, theta_ranges_2 in receiver_band_items:
        print(f"\n############################")
        print(f"# Receiver band: {receiver_band}")
        print(f"############################")

        # 1. Noise varying: noise berubah, fading tetap.
        for scenario_name, noise_values in NOISE_VARYING_SCENARIOS.items():
            manifest.append(generate_one_scenario(
                scenario_type="noise_varying",
                scenario_name=scenario_name,
                noise_var=noise_values,
                fading=FIXED_FADING_FOR_NTV,
                fading_source="fixed_fading_from_NTV_notebook",
                receiver_band=receiver_band,
                theta_ranges_2=theta_ranges_2,
            ))

        # 2. Fading varying: fading berubah, noise tetap.
        for fading_file in FADING_FILES:
            fading_path = BASE_DIR / fading_file
            if not fading_path.exists():
                print(f"SKIP: {fading_path} tidak ditemukan.")
                continue

            fading = load_fading_csv(fading_path)
            scenario_name = Path(fading_file).stem
            manifest.append(generate_one_scenario(
                scenario_type="fading_varying",
                scenario_name=scenario_name,
                noise_var=FIXED_NOISE_VAR_FOR_FTV,
                fading=fading,
                fading_source=fading_file,
                receiver_band=receiver_band,
                theta_ranges_2=theta_ranges_2,
            ))

        # 3. Noise and fading varying: noise berubah, fading berubah.
        for fading_file in FADING_FILES:
            fading_path = BASE_DIR / fading_file
            if not fading_path.exists():
                print(f"SKIP: {fading_path} tidak ditemukan.")
                continue

            fading = load_fading_csv(fading_path)
            scenario_name = Path(fading_file).stem
            manifest.append(generate_one_scenario(
                scenario_type="noise_and_fading_varying",
                scenario_name=scenario_name,
                noise_var=NOISE_VAR_FOR_NTV_FTV,
                fading=fading,
                fading_source=fading_file,
                receiver_band=receiver_band,
                theta_ranges_2=theta_ranges_2,
            ))

    manifest_path = OUTPUT_DIR / "manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nManifest disimpan ke: {manifest_path}")
    print(f"Total entri manifest: {len(manifest)}")
    return manifest


### Jalankan pembuatan dataset

Cell berikut dipisah menjadi dua bagian:

1. `print_dataset_plan()` untuk mengecek jumlah data dan estimasi waktu.
2. `generate_all_datasets()` untuk benar-benar membuat seluruh file dataset.

Default notebook ini memakai proteksi `OVERWRITE_DATASETS = False`. Jika runtime terputus di tengah jalan, jalankan ulang cell generate; skenario yang file-nya sudah lengkap akan dilewati otomatis.


In [ ]:
# Cek rencana jumlah dataset dan estimasi waktu.
print_dataset_plan(minutes_per_scenario=2.5)

# FINAL: dibuat True agar notebook bisa dijalankan sekali dari atas ke bawah.
# Jika runtime terputus, jalankan ulang saja. File yang sudah ada akan dilewati karena OVERWRITE_DATASETS=False.
RUN_DATASET_GENERATION = True

if RUN_DATASET_GENERATION:
    manifest = generate_all_datasets()
else:
    print("Dataset belum dibuat. Ubah RUN_DATASET_GENERATION=True untuk menjalankan generate_all_datasets().")


## 7. Fungsi Load Dataset

Gunakan fungsi ini untuk membaca dataset per-skenario atau menggabungkan beberapa file `.npz` menjadi satu dataset pelatihan.


In [ ]:
def load_npz_dataset(path):
    data = np.load(path, allow_pickle=False)
    X = data["X"]
    y = data["y"]
    metadata = json.loads(data["metadata"].item())
    return X, y, metadata


def load_manifest(manifest_path=OUTPUT_DIR / "manifest.json"):
    with open(manifest_path, "r", encoding="utf-8") as f:
        return json.load(f)


def select_dataset_files(manifest, model_type, scenario_types=None, scenario_names=None, receiver_bands=None):
    if model_type not in {"cnn", "cnn_lstm"}:
        raise ValueError("model_type harus 'cnn' atau 'cnn_lstm'")

    selected = []
    for item in manifest:
        if scenario_types is not None and item["scenario_type"] not in scenario_types:
            continue
        if scenario_names is not None and item["scenario_name"] not in scenario_names:
            continue
        if receiver_bands is not None and item["receiver_band"] not in receiver_bands:
            continue
        selected.append(item[f"{model_type}_path"])

    if not selected:
        raise ValueError("Tidak ada file dataset yang cocok dengan filter yang diberikan.")

    return selected


def load_and_concat(paths):
    X_list, y_list, metadata_list = [], [], []
    for path in paths:
        X, y, metadata = load_npz_dataset(path)
        X_list.append(X)
        y_list.append(y)
        metadata_list.append(metadata)

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_all, y_all, metadata_list


## 7.1 Validasi Manifest dan Dataset

Gunakan cell ini setelah generate dataset selesai. Fungsinya untuk memastikan jumlah file dan jumlah data sudah sesuai dengan rencana.


In [ ]:
def validate_manifest_and_files(manifest):
    expected = expected_dataset_count()
    print("Jumlah entri manifest:", len(manifest))
    print("Ekspektasi skenario dataset:", expected["scenario_count"])

    missing = []
    cnn_samples = 0
    cnn_lstm_samples = 0

    for item in manifest:
        cnn_path = Path(item["cnn_path"])
        lstm_path = Path(item["cnn_lstm_path"])

        if not cnn_path.exists():
            missing.append(str(cnn_path))
        else:
            with np.load(cnn_path, allow_pickle=False) as data:
                cnn_samples += data["X"].shape[0]

        if not lstm_path.exists():
            missing.append(str(lstm_path))
        else:
            with np.load(lstm_path, allow_pickle=False) as data:
                cnn_lstm_samples += data["X"].shape[0]

    print("Data CNN ditemukan     :", f"{cnn_samples:,}".replace(',', '.'))
    print("Data CNN-LSTM ditemukan:", f"{cnn_lstm_samples:,}".replace(',', '.'))

    if missing:
        print("File hilang:")
        for path in missing[:20]:
            print(" -", path)
        if len(missing) > 20:
            print(f"... dan {len(missing) - 20} file lain")
    else:
        print("Semua file pada manifest ditemukan.")

    return {
        "manifest_entries": len(manifest),
        "cnn_samples": cnn_samples,
        "cnn_lstm_samples": cnn_lstm_samples,
        "missing_files": missing,
    }


# Validasi otomatis setelah dataset dibuat atau manifest tersedia.
if 'manifest' not in globals():
    manifest = load_manifest()

validation = validate_manifest_and_files(manifest)


## 8. Model CNN dan CNN-LSTM untuk Grid Search

Fungsi model dibuat parametrik agar dapat dipakai oleh grid search. Grid search di sini bukan untuk mengubah metodologi penelitian utama, melainkan untuk mencari kombinasi parameter pelatihan dan arsitektur yang paling stabil pada dataset gabungan.


In [ ]:
def make_optimizer(name, learning_rate):
    name = name.lower()
    if name == "adam":
        return optimizers.Adam(learning_rate=learning_rate)
    if name == "sgd":
        return optimizers.SGD(learning_rate=learning_rate)
    if name == "rmsprop":
        return optimizers.RMSprop(learning_rate=learning_rate)
    raise ValueError(f"Optimizer tidak dikenali: {name}")


def build_cnn_model(input_shape, filters=(32, 64), kernel_size=3, dense_units=128, pooling="max", dropout=0.0,
                    optimizer_name="adam", learning_rate=1e-3):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    for f in filters:
        x = layers.Conv2D(f, (kernel_size, kernel_size), activation="relu", padding="same")(x)
        x = layers.MaxPooling2D((2, 2))(x)

    if pooling == "gap":
        x = layers.GlobalAveragePooling2D()(x)
    elif pooling == "flatten":
        x = layers.Flatten()(x)
    else:
        raise ValueError("pooling harus 'gap' atau 'flatten'")

    x = layers.Dense(dense_units, activation="relu")(x)
    # if dropout > 0:
    #     x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(2, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_cnn_encoder(input_shape, encoder_filters=(16, 32), kernel_size=3, encoder_pooling="gap"):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    for f in encoder_filters:
        x = layers.Conv2D(f, (kernel_size, kernel_size), activation="relu", padding="same", strides=2)(x)

    if encoder_pooling == "gap":
        x = layers.GlobalAveragePooling2D()(x)
    elif encoder_pooling == "flatten":
        x = layers.Flatten()(x)
    else:
        raise ValueError("encoder_pooling harus 'gap' atau 'flatten'")

    return models.Model(inputs, x)


def build_cnn_lstm_model(input_shape, encoder_filters=(16, 32), kernel_size=3, lstm_units=32, dense_units=0,
                         encoder_pooling="gap", dropout=0.0, optimizer_name="adam", learning_rate=1e-3):
    # input_shape: (time_steps, height, width, channels)
    time_steps = input_shape[0]
    frame_shape = input_shape[1:]

    encoder = build_cnn_encoder(
        input_shape=frame_shape,
        encoder_filters=encoder_filters,
        kernel_size=kernel_size,
        encoder_pooling=encoder_pooling,
    )

    inputs = layers.Input(shape=input_shape)
    x = layers.TimeDistributed(encoder)(inputs)
    x = layers.LSTM(lstm_units, return_sequences=False)(x)

    if dense_units and dense_units > 0:
        x = layers.Dense(dense_units, activation="relu")(x)

    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(2, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


## 9. Manual Grid Search

Grid search dibuat manual agar tidak bergantung pada library tambahan seperti SciKeras atau Keras Tuner. Metrik utama menggunakan `val_accuracy`. Setelah kombinasi terbaik ditemukan, model terbaik dievaluasi pada test set.

Pembagian data default pada fungsi `manual_grid_search()` adalah:

```text
Train      = 70%
Validation = 10%
Test       = 20%
```

Pembagian dilakukan secara stratified sehingga proporsi label 1 (PU + noise) dan label 0 (noise only) tetap seimbang pada train, validation, dan test.


In [ ]:
def detection_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    pd_value = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pfa_value = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    youden_index = pd_value - pfa_value
    return pd_value, pfa_value, youden_index


def json_safe(value):
    """Mengubah object NumPy/Path/tuple agar aman disimpan ke JSON."""
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def expand_param_grid(param_grid):
    """
    Mendukung dua bentuk grid:
    1) dict of lists  -> seluruh kombinasi kartesian seperti grid search biasa.
    2) list of dicts  -> kombinasi eksplisit/terarah, hanya konfigurasi yang ditulis.

    Untuk eksperimen ini, bentuk list of dicts dipakai agar CNN hanya melatih
    3 konfigurasi yang diinginkan dan CNN-LSTM juga hanya 3 konfigurasi.
    """
    if isinstance(param_grid, list):
        for params in param_grid:
            yield dict(params)
        return

    if isinstance(param_grid, dict):
        keys = list(param_grid.keys())
        values = [param_grid[k] for k in keys]
        for combo in product(*values):
            yield dict(zip(keys, combo))
        return

    raise TypeError("param_grid harus berupa dict of lists atau list of dicts")


def count_param_combinations(param_grid):
    if isinstance(param_grid, list):
        return len(param_grid)
    total = 1
    for values in param_grid.values():
        total *= len(values)
    return total


def print_grid_search_plan(minutes_per_training=2.0):
    cnn_runs = count_param_combinations(CNN_PARAM_GRID)
    lstm_runs = count_param_combinations(CNN_LSTM_PARAM_GRID)
    total_runs = cnn_runs + lstm_runs
    total_minutes = total_runs * minutes_per_training
    print("Rencana grid search dataset gabungan besar:")
    print(f"  CNN      : {cnn_runs} training")
    print(f"  CNN-LSTM : {lstm_runs} training")
    print(f"  Total    : {total_runs} training")
    print(f"  Estimasi : {total_minutes:.1f} menit / {total_minutes/60:.2f} jam")
    return {"cnn_runs": cnn_runs, "cnn_lstm_runs": lstm_runs, "total_runs": total_runs, "total_minutes": total_minutes}


def make_global_split_indices(y, train_size=0.7, val_size=0.1, test_size=0.2, random_state=42):
    """
    Membuat indeks train/validation/test dari dataset gabungan besar.

    Catatan penting:
    - Split dilakukan pada indeks global, bukan langsung pada X.
    - Indeks ini kemudian dipakai ulang untuk:
      1) training grid search,
      2) evaluasi test global,
      3) inference/metrik per skenario.
    Dengan demikian, data test per skenario berasal dari test set yang sama persis
    dengan test set yang dipakai saat evaluasi model setelah training.
    """
    total_split = train_size + val_size + test_size
    if not np.isclose(total_split, 1.0):
        raise ValueError(f"train_size + val_size + test_size harus = 1.0, tetapi sekarang = {total_split}")

    indices = np.arange(len(y))

    idx_train_val, idx_test, y_train_val, y_test = train_test_split(
        indices,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    val_size_relative = val_size / (train_size + val_size)
    idx_train, idx_val, y_train, y_val = train_test_split(
        idx_train_val,
        y_train_val,
        test_size=val_size_relative,
        stratify=y_train_val,
        random_state=random_state,
    )

    split_indices = {
        "idx_train": idx_train,
        "idx_val": idx_val,
        "idx_test": idx_test,
    }

    print("Pembagian data:")
    print(f"  Train      : {len(idx_train):,} data ({len(idx_train) / len(y) * 100:.1f}%)")
    print(f"  Validation : {len(idx_val):,} data ({len(idx_val) / len(y) * 100:.1f}%)")
    print(f"  Test       : {len(idx_test):,} data ({len(idx_test) / len(y) * 100:.1f}%)")

    return split_indices


def save_global_split_indices(split_indices, model_type, output_dir=OUTPUT_DIR):
    split_path = output_dir / f"global_split_indices_{model_type}.npz"
    np.savez_compressed(
        split_path,
        idx_train=split_indices["idx_train"],
        idx_val=split_indices["idx_val"],
        idx_test=split_indices["idx_test"],
        seed=SEED,
        train_size=0.7,
        val_size=0.1,
        test_size=0.2,
    )
    print(f"Split indices {model_type} disimpan ke:", split_path)
    return split_path


def evaluate_model_on_split(model, X_split, y_split, split_name="test"):
    """
    Evaluasi model pada split tertentu.

    split_name menentukan prefix key output, misalnya:
    - split_name='val'  -> val_loss, val_accuracy, val_pd, val_pfa, val_youden_index
    - split_name='test' -> test_loss, test_accuracy, test_pd, test_pfa, test_youden_index
    """
    loss, acc = model.evaluate(X_split, y_split, verbose=0)
    y_prob_raw = model.predict(X_split, verbose=0)

    if y_prob_raw.ndim == 2 and y_prob_raw.shape[1] == 2:
        y_prob = y_prob_raw[:, 1]
        y_pred = np.argmax(y_prob_raw, axis=1)
    elif y_prob_raw.ndim == 2 and y_prob_raw.shape[1] == 1:
        y_prob = y_prob_raw[:, 0]
        y_pred = (y_prob >= 0.5).astype(int)
    elif y_prob_raw.ndim == 1:
        y_prob = y_prob_raw
        y_pred = (y_prob >= 0.5).astype(int)
    else:
        raise ValueError(f"Bentuk output model tidak dikenali: {y_prob_raw.shape}")

    cm = confusion_matrix(y_split, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    pd_value, pfa_value, youden_index = detection_metrics(y_split, y_pred)
    fpr, tpr, thresholds = roc_curve(y_split, y_prob)
    roc_auc = auc(fpr, tpr)

    return {
        f"{split_name}_loss": float(loss),
        f"{split_name}_accuracy": float(acc),
        f"{split_name}_pd": float(pd_value),
        f"{split_name}_pfa": float(pfa_value),
        f"{split_name}_youden_index": float(youden_index),
        f"{split_name}_tn": int(tn),
        f"{split_name}_fp": int(fp),
        f"{split_name}_fn": int(fn),
        f"{split_name}_tp": int(tp),
        f"{split_name}_size": int(len(y_split)),
        "y_pred": y_pred,
        "y_prob": y_prob,
        "fpr": fpr,
        "tpr": tpr,
        "roc_auc": roc_auc,
    }


def evaluate_model_on_test_set(model, X_test, y_test):
    """
    Evaluasi model pada test set global.

    Fungsi ini mempertahankan key lama: test_loss, test_accuracy, pd, pfa, youden_index.
    Selain itu, key baru juga tersedia: test_pd, test_pfa, test_youden_index.
    """
    metrics = evaluate_model_on_split(model, X_test, y_test, split_name="test")
    metrics.update({
        "pd": metrics["test_pd"],
        "pfa": metrics["test_pfa"],
        "youden_index": metrics["test_youden_index"],
        "tn": metrics["test_tn"],
        "fp": metrics["test_fp"],
        "fn": metrics["test_fn"],
        "tp": metrics["test_tp"],
        "test_size": metrics["test_size"],
    })
    return metrics


def save_config_result_json(model_type, run_index, run_payload, output_dir=OUTPUT_DIR):
    """Menyimpan hasil validasi dan test global per konfigurasi ke file JSON terpisah."""
    config_dir = output_dir / f"per_config_test_results_{model_type}"
    config_dir.mkdir(parents=True, exist_ok=True)
    json_path = config_dir / f"{model_type}_run_{run_index:02d}_metrics.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_safe(run_payload), f, indent=2)

    print("JSON hasil konfigurasi disimpan ke:", json_path)
    return json_path


def manual_grid_search(X, y, model_type, param_grid, train_size=0.7, val_size=0.1, test_size=0.2, random_state=42):
    # Split dibuat sekali, lalu dipakai untuk semua kombinasi hyperparameter.
    split_indices = make_global_split_indices(
        y,
        train_size=train_size,
        val_size=val_size,
        test_size=test_size,
        random_state=random_state,
    )

    idx_train = split_indices["idx_train"]
    idx_val = split_indices["idx_val"]
    idx_test = split_indices["idx_test"]

    X_train, y_train = X[idx_train], y[idx_train]
    X_val, y_val = X[idx_val], y[idx_val]
    X_test, y_test = X[idx_test], y[idx_test]

    # Simpan indeks agar inference/metrik per skenario tidak perlu merekonstruksi split.
    save_global_split_indices(split_indices, model_type=model_type)

    results = []
    best = None
    
    # --- TAMBAHAN: Inisialisasi list penyimpan data ROC ---
    roc_data = [] 
    # ------------------------------------------------------
    
    param_combinations = list(expand_param_grid(param_grid))

    for run_index, raw_params in enumerate(param_combinations, start=1):
        params = dict(raw_params)
        print(f"\n=== Grid run {run_index}/{len(param_combinations)} | {model_type} ===")
        print(params)

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)

        epochs = params.pop("epochs")
        batch_size = params.pop("batch_size")

        fit_kwargs = {
            "epochs": epochs,
            "batch_size": batch_size,
            "verbose": 1,
            "callbacks": [
                callbacks.EarlyStopping(
                    monitor="val_accuracy",
                    patience=5,
                    mode="max",
                    restore_best_weights=True,
                )
            ],
        }

        if model_type == "cnn":
            model = build_cnn_model(input_shape=X.shape[1:], **params)
        elif model_type == "cnn_lstm":
            model = build_cnn_lstm_model(input_shape=X.shape[1:], **params)
        else:
            raise ValueError("model_type harus 'cnn' atau 'cnn_lstm'")

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            **fit_kwargs,
        )

        history_train_acc = max(history.history["accuracy"])
        history_val_acc = max(history.history["val_accuracy"])
        history_val_loss = min(history.history["val_loss"])
        best_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)

        # Evaluasi validation dan test global untuk SETIAP konfigurasi setelah bobot terbaik dipulihkan.
        val_metrics = evaluate_model_on_split(model, X_val, y_val, split_name="val")
        test_metrics = evaluate_model_on_test_set(model, X_test, y_test)

        # --- TAMBAHAN: Menyimpan hasil ROC berdasarkan Test Set ---
        roc_data.append({
            "run_index": run_index,
            "fpr": test_metrics["fpr"],
            "tpr": test_metrics["tpr"],
            "roc_auc": test_metrics["roc_auc"],
            "label_params": str(raw_params)
        })
        # ----------------------------------------------------------

        # --- TAMBAHAN: Kecualikan fpr dan tpr agar tidak masuk ke file JSON hasil config ---
        val_metrics_to_save = {k: v for k, v in val_metrics.items() if k not in {"y_pred", "y_prob", "fpr", "tpr"}}
        test_metrics_to_save = {k: v for k, v in test_metrics.items() if k not in {"y_pred", "y_prob", "fpr", "tpr"}}

        selection_metric_name = "val_accuracy"
        selection_metric_value = val_metrics_to_save["val_accuracy"]

        run_payload = {
            "run_index": run_index,
            "model_type": model_type,
            "selection_basis": selection_metric_name,
            "selection_metric_value": float(selection_metric_value),
            "params": raw_params,
            "training_history_summary": {
                "best_epoch_by_val_accuracy": best_epoch,
                "max_train_accuracy": float(history_train_acc),
                "max_val_accuracy_history": float(history_val_acc),
                "min_val_loss_history": float(history_val_loss),
                "epochs_requested": int(epochs),
                "epochs_ran": int(len(history.history["loss"])),
                "batch_size": int(batch_size),
            },
            "validation_metrics": val_metrics_to_save,
            "test_metrics": test_metrics_to_save,
            **val_metrics_to_save,
            **test_metrics_to_save,
        }
        config_json_path = save_config_result_json(model_type, run_index, run_payload)

        row = {
            "run_index": run_index,
            "model_type": model_type,
            "selection_basis": selection_metric_name,
            "selection_metric_value": float(selection_metric_value),
            "history_train_accuracy": float(history_train_acc),
            "history_val_accuracy": float(history_val_acc),
            "history_val_loss": float(history_val_loss),
            "best_epoch": best_epoch,
            "epochs_requested": int(epochs),
            "epochs_ran": int(len(history.history["loss"])),
            "batch_size": int(batch_size),
            "config_json_path": str(config_json_path),
            **val_metrics_to_save,
            **test_metrics_to_save,
            **raw_params,
        }
        results.append(row)

        if best is None or selection_metric_value > best["selection_metric_value"]:
            best = {
                "selection_basis": selection_metric_name,
                "selection_metric_value": float(selection_metric_value),
                "val_accuracy": float(val_metrics_to_save["val_accuracy"]),
                "val_pd": float(val_metrics_to_save["val_pd"]),
                "val_pfa": float(val_metrics_to_save["val_pfa"]),
                "val_youden_index": float(val_metrics_to_save["val_youden_index"]),
                "model": model,
                "params": raw_params,
                "history": history,
                "split_indices": split_indices,
                "best_epoch": best_epoch,
                "val_metrics": val_metrics_to_save,
                "test_metrics": test_metrics_to_save,
                "config_json_path": str(config_json_path),
            }

    # --- TAMBAHAN: Plotting Keseluruhan ROC ---
    plt.figure(figsize=(10, 8))
    for data in roc_data:
        plt.plot(
            data["fpr"], 
            data["tpr"], 
            label=f'Run {data["run_index"]} (AUC = {data["roc_auc"]:.4f})'
        )
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
    plt.xlabel('False Positive Rate (Pfa)')
    plt.ylabel('True Positive Rate (Pd)')
    plt.title(f'ROC Curves for all Grid Search Configurations ({model_type.upper()})')
    plt.legend(loc='lower right', fontsize='small')
    plt.grid(True)
    
    # Opsional: Menyimpan grafik ke file
    roc_plot_path = OUTPUT_DIR / f"roc_curves_{model_type}.png"
    plt.savefig(roc_plot_path)
    print(f"\nGrafik ROC gabungan disimpan ke: {roc_plot_path}")
    
    plt.show()
    # ------------------------------------------

    results_df = pd.DataFrame(results).sort_values("selection_metric_value", ascending=False)
    return results_df, best


def evaluate_best_model(best, X, y):
    model = best["model"]
    idx_test = best["split_indices"]["idx_test"]

    X_test = X[idx_test]
    y_test = y[idx_test]

    eval_metrics = evaluate_model_on_test_set(model, X_test, y_test)
    y_pred = eval_metrics["y_pred"]
    y_prob = eval_metrics["y_prob"]

    print("Selection basis:", best.get("selection_basis"))
    print("Selection metric value:", best.get("selection_metric_value"))
    print("Validation accuracy:", best.get("val_accuracy"))
    print("Validation Pd:", best.get("val_pd"))
    print("Validation Pfa:", best.get("val_pfa"))
    print("Validation Youden Index:", best.get("val_youden_index"))
    print("Test loss:", eval_metrics["test_loss"])
    print("Test accuracy:", eval_metrics["test_accuracy"])
    print("Test Pd:", eval_metrics["test_pd"])
    print("Test Pfa:", eval_metrics["test_pfa"])
    print("Test Youden Index:", eval_metrics["test_youden_index"])
    print("Best params:", best["params"])
    print("Best epoch:", best["best_epoch"])
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

    return {
        "selection_basis": best.get("selection_basis"),
        "selection_metric_value": float(best.get("selection_metric_value")),
        "val_loss": float(best["val_metrics"]["val_loss"]),
        "val_accuracy": float(best["val_metrics"]["val_accuracy"]),
        "val_pd": float(best["val_metrics"]["val_pd"]),
        "val_pfa": float(best["val_metrics"]["val_pfa"]),
        "val_youden_index": float(best["val_metrics"]["val_youden_index"]),
        "test_loss": float(eval_metrics["test_loss"]),
        "test_accuracy": float(eval_metrics["test_accuracy"]),
        "test_pd": float(eval_metrics["test_pd"]),
        "test_pfa": float(eval_metrics["test_pfa"]),
        "test_youden_index": float(eval_metrics["test_youden_index"]),
        # Key lama dipertahankan sebagai alias metrik test.
        "pd": float(eval_metrics["test_pd"]),
        "pfa": float(eval_metrics["test_pfa"]),
        "youden_index": float(eval_metrics["test_youden_index"]),
        "tn": int(eval_metrics["test_tn"]),
        "fp": int(eval_metrics["test_fp"]),
        "fn": int(eval_metrics["test_fn"]),
        "tp": int(eval_metrics["test_tp"]),
        "y_pred": y_pred,
        "y_prob": y_prob,
        "best_params": best["params"],
        "best_epoch": best["best_epoch"],
        "config_json_path": best.get("config_json_path"),
        "test_size": int(len(y_test)),
    }


## 9.1 Evaluasi Test Per Skenario dengan Split yang Sama dari Training

Bagian ini menggabungkan logika inference ke notebook training. Metrik per skenario dihitung memakai `idx_test` yang sama persis dengan split yang dipakai saat grid search.


In [ ]:
def build_dataset_table(manifest, model_type):
    """
    Mengubah manifest menjadi tabel dataset per skenario untuk model tertentu.
    Tabel ini menjaga urutan file persis seperti proses load dataset gabungan.
    """
    if model_type not in {"cnn", "cnn_lstm"}:
        raise ValueError("model_type harus 'cnn' atau 'cnn_lstm'")

    rows = []
    path_key = f"{model_type}_path"

    for item_index, item in enumerate(manifest):
        rows.append({
            "item_index": item_index,
            "model_type": model_type,
            "receiver_band": item["receiver_band"],
            "scenario_type": item["scenario_type"],
            "scenario_name": item["scenario_name"],
            "dataset_path": item[path_key],
        })

    return pd.DataFrame(rows)


def collect_dataset_ranges_from_table(dataset_table):
    """
    Membuat peta indeks global untuk setiap file skenario.

    Contoh:
    file skenario pertama  : indeks global 0-1999
    file skenario kedua    : indeks global 2000-3999
    dst.
    """
    ranges = []
    start = 0

    for _, row in dataset_table.iterrows():
        _, y, _ = load_npz_dataset(row["dataset_path"])
        end = start + len(y)
        ranges.append({
            "item_index": int(row["item_index"]),
            "start": start,
            "end": end,
            "n_total": len(y),
            "dataset_path": row["dataset_path"],
        })
        start = end

    return pd.DataFrame(ranges)


def local_test_indices_for_file(global_test_indices, file_start, file_end):
    """
    Mengambil indeks test global yang berada pada rentang file tertentu,
    lalu mengubahnya menjadi indeks lokal file tersebut.
    """
    global_test_indices = np.asarray(global_test_indices)
    mask = (global_test_indices >= file_start) & (global_test_indices < file_end)
    return global_test_indices[mask] - file_start


def predict_binary(model, X, threshold=0.5, batch_size=256):
    raw = model.predict(X, batch_size=batch_size, verbose=0)

    # Model yang dipakai di notebook ini memakai Dense(2, activation='softmax').
    # Jika suatu saat diganti menjadi sigmoid 1 output, fungsi ini tetap aman.
    if raw.ndim == 2 and raw.shape[1] == 2:
        proba = raw[:, 1]
    elif raw.ndim == 2 and raw.shape[1] == 1:
        proba = raw[:, 0]
    elif raw.ndim == 1:
        proba = raw
    else:
        raise ValueError(f"Bentuk output model tidak dikenali: {raw.shape}")

    y_pred = (proba >= threshold).astype(int)
    return y_pred, proba


def compute_binary_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    pd_value = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pfa_value = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    youden_index = pd_value - pfa_value

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "pd": pd_value,
        "pfa": pfa_value,
        "youden_index": youden_index,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support_0_noise_only": int(tn + fp),
        "support_1_pu_noise": int(tp + fn),
    }


def evaluate_one_dataset_file_from_training_split(
    model,
    row,
    ranges_df,
    split_indices,
    threshold=0.5,
):
    """
    Evaluasi satu file skenario memakai idx_test yang sama persis dari training.
    """
    X, y, metadata = load_npz_dataset(row["dataset_path"])

    range_row = ranges_df.loc[ranges_df["item_index"] == int(row["item_index"])].iloc[0]
    eval_indices = local_test_indices_for_file(
        split_indices["idx_test"],
        int(range_row["start"]),
        int(range_row["end"]),
    )

    if len(eval_indices) == 0:
        raise ValueError(f"Tidak ada data test untuk file: {row['dataset_path']}")

    X_eval = X[eval_indices]
    y_eval = y[eval_indices]
    y_pred, proba = predict_binary(model, X_eval, threshold=threshold)
    metrics = compute_binary_metrics(y_eval, y_pred)

    result = {
        "model_type": row["model_type"],
        "receiver_band": row["receiver_band"],
        "scenario_type": row["scenario_type"],
        "scenario_name": row["scenario_name"],
        "evaluation_mode": "training_test_split",
        "threshold": threshold,
        "n_total_file": int(len(y)),
        "n_eval": int(len(y_eval)),
        "dataset_path": row["dataset_path"],
    }
    result.update(metrics)
    return result


def evaluate_model_per_scenario_from_training_split(
    model,
    dataset_table,
    ranges_df,
    split_indices,
    threshold=0.5,
):
    rows = []

    for i, (_, row) in enumerate(dataset_table.iterrows(), start=1):
        print(f"[{i}/{len(dataset_table)}] {row['model_type']} | {row['receiver_band']} | {row['scenario_type']} | {row['scenario_name']}")
        result = evaluate_one_dataset_file_from_training_split(
            model=model,
            row=row,
            ranges_df=ranges_df,
            split_indices=split_indices,
            threshold=threshold,
        )
        rows.append(result)

    return pd.DataFrame(rows)


def export_inference_results(results_df, evaluation_mode="training_test_split", output_dir=OUTPUT_DIR):
    if results_df is None or results_df.empty:
        print("Tidak ada hasil inference untuk disimpan.")
        return None

    inference_csv_path = output_dir / f"inference_metrics_per_scenario__{evaluation_mode}.csv"
    inference_xlsx_path = output_dir / f"inference_metrics_per_scenario__{evaluation_mode}.xlsx"

    results_df.to_csv(inference_csv_path, index=False)
    print("CSV per skenario disimpan ke :", inference_csv_path)

    try:
        results_df.to_excel(inference_xlsx_path, index=False)
        print("Excel per skenario disimpan ke:", inference_xlsx_path)
    except Exception as err:
        print("Excel per skenario tidak berhasil disimpan. CSV tetap tersedia.")
        print("Error:", err)

    metric_columns = ["accuracy", "precision", "recall", "f1_score", "pd", "pfa", "youden_index"]

    summary_by_model = results_df.groupby("model_type")[metric_columns].mean().reset_index()
    summary_by_pu = results_df.groupby(["model_type", "receiver_band"])[metric_columns].mean().reset_index()
    summary_by_type = results_df.groupby(["model_type", "scenario_type"])[metric_columns].mean().reset_index()
    summary_by_pu_type = results_df.groupby(["model_type", "receiver_band", "scenario_type"])[metric_columns].mean().reset_index()

    summary_by_model.to_csv(output_dir / f"inference_summary_by_model__{evaluation_mode}.csv", index=False)
    summary_by_pu.to_csv(output_dir / f"inference_summary_by_pu__{evaluation_mode}.csv", index=False)
    summary_by_type.to_csv(output_dir / f"inference_summary_by_scenario_type__{evaluation_mode}.csv", index=False)
    summary_by_pu_type.to_csv(output_dir / f"inference_summary_by_pu_and_scenario_type__{evaluation_mode}.csv", index=False)

    summary_xlsx_path = output_dir / f"inference_metrics_and_summaries__{evaluation_mode}.xlsx"
    try:
        with pd.ExcelWriter(summary_xlsx_path) as writer:
            results_df.to_excel(writer, sheet_name="per_scenario", index=False)
            summary_by_model.to_excel(writer, sheet_name="summary_by_model", index=False)
            summary_by_pu.to_excel(writer, sheet_name="summary_by_pu", index=False)
            summary_by_type.to_excel(writer, sheet_name="summary_by_type", index=False)
            summary_by_pu_type.to_excel(writer, sheet_name="summary_by_pu_type", index=False)
        print("Excel gabungan disimpan ke:", summary_xlsx_path)
    except Exception as err:
        print("Excel gabungan tidak berhasil disimpan. Seluruh CSV summary tetap tersedia.")
        print("Error:", err)

    print("\nRata-rata per model:")
    display(summary_by_model)

    print("\nRata-rata per PU:")
    display(summary_by_pu)

    print("\nRata-rata per jenis skenario:")
    display(summary_by_type)

    print("\nRata-rata per PU dan jenis skenario:")
    display(summary_by_pu_type)

    return {
        "per_scenario": results_df,
        "summary_by_model": summary_by_model,
        "summary_by_pu": summary_by_pu,
        "summary_by_type": summary_by_type,
        "summary_by_pu_type": summary_by_pu_type,
    }


## 10. Grid Search pada Dataset Gabungan Besar

Bagian ini adalah skema utama yang direkomendasikan: semua skenario dan semua PU digabung menjadi satu dataset besar untuk masing-masing arsitektur.

Dengan konfigurasi default:

```text
CNN      = 108.000 data
CNN-LSTM = 108.000 data
```

Pembagian data untuk masing-masing arsitektur adalah:

```text
Train      = 70% = 75.600 data
Validation = 10% = 10.800 data
Test       = 20% = 21.600 data
```

Karena dataset seimbang, pada setiap format input jumlah labelnya adalah:

```text
Train      = 37.800 PU + noise dan 37.800 noise only
Validation = 5.400 PU + noise dan 5.400 noise only
Test       = 10.800 PU + noise dan 10.800 noise only
```

Grid search dijalankan sekali untuk CNN dan sekali untuk CNN-LSTM. Dengan grid eksplisit saat ini, jumlah training adalah 3 konfigurasi CNN + 3 konfigurasi CNN-LSTM = 6 training.

Setiap konfigurasi akan menghasilkan satu file JSON berisi `test_loss`, `test_accuracy`, `pd`, `pfa`, dan `youden_index` pada test set global.



In [ ]:
# Grid eksplisit/terarah.
# Setiap dict = satu konfigurasi yang akan dilatih.
# Format CNN: filters layer 1-2 dan dense_units.
# Format CNN-LSTM: encoder_filters layer 1-2 dan lstm_units.

# CNN_PARAM_GRID = [
#     {
#         "filters": (32, 64),
#         "kernel_size": 3,
#         "dense_units": 128,
#         "pooling": "flatten",
#         "optimizer_name": "adam",
#         "learning_rate": 1e-3,
#         "epochs": 35,
#         "batch_size": 256,
#     },
#     {
#         "filters": (32, 64),
#         "kernel_size": 3,
#         "dense_units": 256,
#         "pooling": "flatten",
#         "optimizer_name": "adam",
#         "learning_rate": 1e-3,
#         "epochs": 35,
#         "batch_size": 256,
#     },
#     {
#         "filters": (64, 128),
#         "kernel_size": 3,
#         "dense_units": 128,
#         "pooling": "flatten",
#         "optimizer_name": "adam",
#         "learning_rate": 1e-3,
#         "epochs": 35,
#         "batch_size": 256,
#     },
# ]

# CNN_LSTM_PARAM_GRID = [
#     {
#         "encoder_filters": (8, 16),
#         "kernel_size": 3,
#         "lstm_units": 32,
#         "encoder_pooling": "flatten",
#         "dropout": 0.1,
#         "optimizer_name": "adam",
#         "learning_rate": 1e-3,
#         "epochs": 35,
#         "batch_size": 256,
#     },
#     {
#         "encoder_filters": (8, 16),
#         "kernel_size": 3,
#         "lstm_units": 64,
#         "encoder_pooling": "flatten",
#         "dropout": 0.1,
#         "optimizer_name": "adam",
#         "learning_rate": 1e-3,
#         "epochs": 35,
#         "batch_size": 256,
#     },
#     {
#         "encoder_filters": (16, 32),
#         "kernel_size": 3,
#         "lstm_units": 32,
#         "encoder_pooling": "flatten",
#         "dropout": 0.1,
#         "optimizer_name": "adam",
#         "learning_rate": 1e-3,
#         "epochs": 35,
#         "batch_size": 256,
#     },
# ]

# CNN_PARAM_GRID = {
#     "filters": [(32,64),(64,128)],
#     "kernel_size": [3],
#     "dense_units": [128,256],
#     "pooling": ["flatten"],
#     # "dropout": [0.0, 0.2],
#     "optimizer_name": ["adam"],
#     "learning_rate": [1e-3],
#     "epochs": [35],
#     "batch_size": [256],
# }

# CNN_LSTM_PARAM_GRID = {
#     "encoder_filters": [(8,16),(16, 32)],
#     "kernel_size": [3],
#     "lstm_units": [32,64],
#     # "dense_units": [0, 64],
#     "encoder_pooling": ["flatten"],
#     "dropout": [0.1],
#     "optimizer_name": ["adam"],
#     "learning_rate": [1e-3],
#     "epochs": [35],
#     "batch_size": [256],
# }

CNN_PARAM_GRID = {
    "filters": [(20,50)],
    "kernel_size": [5],
    "dense_units": [500],
    "pooling": ["flatten"],
    # "dropout": [0.0, 0.2],
    "optimizer_name": ["adam"],
    "learning_rate": [1e-3],
    "epochs": [35],
    "batch_size": [256],
}

CNN_LSTM_PARAM_GRID = {
    "encoder_filters": [(6,6)],
    "kernel_size": [3],
    "lstm_units": [20],
    # "dense_units": [0, 64],
    "encoder_pooling": ["flatten"],
    "dropout": [0.1],
    "optimizer_name": ["sgd"],
    "learning_rate": [1e-3],
    "epochs": [35],
    "batch_size": [256],
}

In [ ]:
# Cek estimasi jumlah training grid search.
print_grid_search_plan(minutes_per_training=2.0)

# FINAL: dibuat True agar notebook bisa dijalankan sekali dari atas ke bawah.
# Jika hanya ingin menjalankan salah satu model, ubah flag yang tidak diperlukan menjadi False.
RUN_GRID_SEARCH_CNN = True
RUN_GRID_SEARCH_CNN_LSTM = True

# FINAL: inference per skenario langsung dilakukan di notebook training.
# Evaluasi ini memakai idx_test yang sama persis dengan split saat grid search.
RUN_TEST_INFERENCE_PER_SCENARIO = True
THRESHOLD = 0.5

all_inference_results = []

if RUN_GRID_SEARCH_CNN or RUN_GRID_SEARCH_CNN_LSTM:
    manifest = load_manifest()

if RUN_GRID_SEARCH_CNN:
    # Dataset gabungan besar: semua PU + semua skenario untuk CNN.
    cnn_files = select_dataset_files(manifest, model_type="cnn")
    X_cnn_all, y_cnn_all, cnn_metadata = load_and_concat(cnn_files)
    print("CNN all dataset:", X_cnn_all.shape, y_cnn_all.shape)

    cnn_results_df, best_cnn = manual_grid_search(
        X_cnn_all,
        y_cnn_all,
        model_type="cnn",
        param_grid=CNN_PARAM_GRID,
    )
    cnn_results_path = OUTPUT_DIR / "grid_search_results_cnn__all_pu_all_scenarios.csv"
    cnn_results_df.to_csv(cnn_results_path, index=False)
    print("Hasil grid search CNN disimpan ke:", cnn_results_path)

    cnn_eval = evaluate_best_model(best_cnn, X_cnn_all, y_cnn_all)
    cnn_model_path = OUTPUT_DIR / "best_cnn__all_pu_all_scenarios.keras"
    best_cnn["model"].save(cnn_model_path)
    print("Model terbaik CNN disimpan ke:", cnn_model_path)

    cnn_eval_to_save = {k: v for k, v in cnn_eval.items() if k not in {"y_pred", "y_prob"}}
    with open(OUTPUT_DIR / "test_evaluation_cnn__all_pu_all_scenarios.json", "w", encoding="utf-8") as f:
        json.dump(cnn_eval_to_save, f, indent=2)

    if RUN_TEST_INFERENCE_PER_SCENARIO:
        print("\nMembuat metrik test per skenario untuk CNN dengan idx_test training yang sama...")
        cnn_table = build_dataset_table(manifest, model_type="cnn")
        cnn_ranges_df = collect_dataset_ranges_from_table(cnn_table)
        cnn_inference_df = evaluate_model_per_scenario_from_training_split(
            model=best_cnn["model"],
            dataset_table=cnn_table,
            ranges_df=cnn_ranges_df,
            split_indices=best_cnn["split_indices"],
            threshold=THRESHOLD,
        )
        all_inference_results.append(cnn_inference_df)
        display(cnn_inference_df.head())

    del X_cnn_all, y_cnn_all, cnn_metadata, best_cnn
    gc.collect()
    tf.keras.backend.clear_session()

if RUN_GRID_SEARCH_CNN_LSTM:
    # Dataset gabungan besar: semua PU + semua skenario untuk CNN-LSTM.
    cnn_lstm_files = select_dataset_files(manifest, model_type="cnn_lstm")
    X_lstm_all, y_lstm_all, lstm_metadata = load_and_concat(cnn_lstm_files)
    print("CNN-LSTM all dataset:", X_lstm_all.shape, y_lstm_all.shape)

    lstm_results_df, best_lstm = manual_grid_search(
        X_lstm_all,
        y_lstm_all,
        model_type="cnn_lstm",
        param_grid=CNN_LSTM_PARAM_GRID,
    )
    lstm_results_path = OUTPUT_DIR / "grid_search_results_cnn_lstm__all_pu_all_scenarios.csv"
    lstm_results_df.to_csv(lstm_results_path, index=False)
    print("Hasil grid search CNN-LSTM disimpan ke:", lstm_results_path)

    lstm_eval = evaluate_best_model(best_lstm, X_lstm_all, y_lstm_all)
    lstm_model_path = OUTPUT_DIR / "best_cnn_lstm__all_pu_all_scenarios.keras"
    best_lstm["model"].save(lstm_model_path)
    print("Model terbaik CNN-LSTM disimpan ke:", lstm_model_path)

    lstm_eval_to_save = {k: v for k, v in lstm_eval.items() if k not in {"y_pred", "y_prob"}}
    with open(OUTPUT_DIR / "test_evaluation_cnn_lstm__all_pu_all_scenarios.json", "w", encoding="utf-8") as f:
        json.dump(lstm_eval_to_save, f, indent=2)

    if RUN_TEST_INFERENCE_PER_SCENARIO:
        print("\nMembuat metrik test per skenario untuk CNN-LSTM dengan idx_test training yang sama...")
        lstm_table = build_dataset_table(manifest, model_type="cnn_lstm")
        lstm_ranges_df = collect_dataset_ranges_from_table(lstm_table)
        lstm_inference_df = evaluate_model_per_scenario_from_training_split(
            model=best_lstm["model"],
            dataset_table=lstm_table,
            ranges_df=lstm_ranges_df,
            split_indices=best_lstm["split_indices"],
            threshold=THRESHOLD,
        )
        all_inference_results.append(lstm_inference_df)
        display(lstm_inference_df.head())

    del X_lstm_all, y_lstm_all, lstm_metadata, best_lstm
    gc.collect()
    tf.keras.backend.clear_session()

if RUN_TEST_INFERENCE_PER_SCENARIO and all_inference_results:
    inference_results_df = pd.concat(all_inference_results, ignore_index=True)
    inference_exports = export_inference_results(
        inference_results_df,
        evaluation_mode="training_test_split",
        output_dir=OUTPUT_DIR,
    )
    display(inference_results_df)
elif RUN_TEST_INFERENCE_PER_SCENARIO:
    print("Tidak ada hasil inference karena grid search tidak dijalankan.")


## 11. Contoh Grid Search Per Kelompok Skenario

Jika tidak ingin menggabungkan semua skenario sekaligus, filter berdasarkan `scenario_type`. Ini sering lebih mudah dianalisis karena hasil CNN/CNN-LSTM dapat dibandingkan pada noise varying, fading varying, dan noise-and-fading varying secara terpisah.


In [ ]:
# manifest = load_manifest()

# Grid search per kelompok skenario, untuk semua PU digabung.
# for scenario_type in ["noise_varying", "fading_varying", "noise_and_fading_varying"]:
#     print("\n=====", scenario_type, "=====")
#     files = select_dataset_files(
#         manifest,
#         model_type="cnn",
#         scenario_types=[scenario_type],
#     )
#     X_group, y_group, _ = load_and_concat(files)
#     results_df, best = manual_grid_search(X_group, y_group, "cnn", CNN_PARAM_GRID)
#     results_df.to_csv(OUTPUT_DIR / f"grid_search_cnn__{scenario_type}__all_pu.csv", index=False)
#     evaluate_best_model(best)

# Grid search per PU dan per kelompok skenario.
# for receiver_band in RECEIVER_BANDS.keys():
#     for scenario_type in ["noise_varying", "fading_varying", "noise_and_fading_varying"]:
#         print("\n=====", receiver_band, scenario_type, "=====")
#         files = select_dataset_files(
#             manifest,
#             model_type="cnn",
#             receiver_bands=[receiver_band],
#             scenario_types=[scenario_type],
#         )
#         X_group, y_group, _ = load_and_concat(files)
#         results_df, best = manual_grid_search(X_group, y_group, "cnn", CNN_PARAM_GRID)
#         results_df.to_csv(OUTPUT_DIR / f"grid_search_cnn__{receiver_band}__{scenario_type}.csv", index=False)
#         evaluate_best_model(best)


## Ringkasan Cara Pakai

Notebook ini sudah dibuat untuk **sekali run** dari atas ke bawah.

Sebelum menjalankan:

1. Pastikan `BASE_DIR` sudah sesuai.
2. Pastikan file `fading-10.csv` sampai `fading-15.csv` tersedia di `BASE_DIR`.
3. Jalankan semua cell dengan `Runtime > Run all`.

Default output:

```text
generated_datasets/
├── 108 file .npz
├── manifest.json
├── global_split_indices_cnn.npz
├── global_split_indices_cnn_lstm.npz
├── grid_search_results_cnn__all_pu_all_scenarios.csv
├── grid_search_results_cnn_lstm__all_pu_all_scenarios.csv
├── per_config_test_results_cnn/*.json
├── per_config_test_results_cnn_lstm/*.json
├── test_evaluation_cnn__all_pu_all_scenarios.json
├── test_evaluation_cnn_lstm__all_pu_all_scenarios.json
├── best_cnn__all_pu_all_scenarios.keras
├── best_cnn_lstm__all_pu_all_scenarios.keras
├── inference_metrics_per_scenario__training_test_split.csv
├── inference_summary_by_model__training_test_split.csv
├── inference_summary_by_pu__training_test_split.csv
├── inference_summary_by_scenario_type__training_test_split.csv
└── inference_summary_by_pu_and_scenario_type__training_test_split.csv
```

File Excel juga akan dibuat jika library `openpyxl` tersedia. Jika `openpyxl` belum terpasang, file CSV tetap tersimpan dan bisa langsung dipakai.

